[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/34.%20The%20Hazard%20%26%20IMDG%20Segregation%20Problem/P34-Tier-8_executed.ipynb)

---


# 34. The Hazard and IMDG Segregation Problem

## Tier 8: A Value-Aligned Ethical Framework

### Goal

This notebook evaluates a small set of container-placement alternatives using safety measures, stakeholder impacts, ethical thresholds, and a weighted ethical score. It illustrates how ethical considerations can be represented explicitly in a decision process. It does not certify a stowage plan or replace the applicable IMDG Code, competent authorities, or professional safety assessment.

### Inputs and outputs

The inputs are four example containers, twelve candidate positions, stakeholder descriptions, ethical constraints, and utility functions. The notebook evaluates three predefined placements, reports their measures and constraint status, ranks eligible alternatives by a weighted ethical score, and produces a visualization and decision report.

### Method in brief

The decision sequence is: define the ethical framework; define the example containers and positions; calculate safety, cost, and stakeholder impacts for each placement; test ethical constraints; calculate weighted utility scores; select the highest-scoring compliant alternative, or rank all alternatives if none is compliant.

The method is a transparent, illustrative multi-criteria decision procedure. The three placements are supplied as alternatives; this notebook does not search all possible assignments or prove that any alternative is globally optimal.

### Model assumptions and scope

- Coordinates are measured in metres, and Euclidean distance is used to measure separation between positions.
- For this example, a 3 m separation threshold is applied to every pair of containers in the listed hazardous classes. This simplified threshold is a teaching assumption, not a statement of actual IMDG segregation requirements.
- Position sensitivity, health risk, environmental risk, operational costs, utility functions, and ethical thresholds are illustrative scoring inputs. They are not empirically validated risk estimates.
- Stakeholder impacts are normalized averages on a scale intended to range from -1 (negative) to +1 (positive). The scores support comparison within this example; they are not measurements of real-world harm or fairness.
- The reported risk-reduction value is a scenario calculation based on an assumed incident cost and an illustrative fraction of safety score. It is not a forecast, a causal estimate, or a verified monetary benefit.
- Ethical principles are represented by explicit numerical measures and thresholds. Such a model can support structured discussion, but it cannot establish that a decision is ethically correct in every context.

### Mathematical structure

For each alternative \(a\), the model calculates a safety score \(S_a\in[0,1]\), ethical utility scores \(U_{ap}\in[0,1]\) for principles \(p\), and an operational cost \(C_a\). The eligible set is

$$
\mathcal{F}=\{a: g_{ap}(a)\geq \tau_p \text{ for every applicable constraint }p\},
$$

where \(g_{ap}(a)\) is the measure used for constraint \(p\), and \(\tau_p\) is its threshold. The selected alternative maximizes the normalized weighted ethical score over eligible alternatives:

$$
a^*\in\operatorname*{arg\,max}_{a\in\mathcal{F}}
\frac{\sum_{p\in P}w_pU_{ap}}{\sum_{p\in P}w_p}.
$$

Here \(P\) is the set of ethical principles with utility functions and \(w_p\) is the corresponding nonnegative weight. If no alternative satisfies every constraint, the notebook explicitly uses all supplied alternatives for ranking and reports that no fully compliant option was found. This fallback is a decision aid, not evidence that the selected alternative meets the constraints.


In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Dict, List, Union

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

## 1. Define the ethical framework

A stakeholder is a group affected by the decision; a principle describes a value the decision process aims to represent. A constraint is a minimum acceptance test, while a utility function assigns a score used to compare alternatives. Keeping constraints and utilities distinct matters: a high weighted score does not, by itself, prove that every minimum requirement has been met.

In the implementation, `EthicalConstraint.threshold` is interpreted in the units of its measurement. For example, the non-maleficence threshold is a safety score, the justice and sustainability thresholds are utility scores, and the precautionary threshold is a separation multiplier. This makes each comparison meaningful rather than comparing unlike quantities.

In [ ]:
class EthicalPrinciple(Enum):
    """Ethical principles considered in the example."""

    NON_MALEFICENCE = "Non-maleficence"
    BENEFICENCE = "Beneficence"
    AUTONOMY = "Autonomy"
    JUSTICE = "Justice"
    PRECAUTIONARY = "Precautionary Principle"
    TRANSPARENCY = "Transparency"
    SUSTAINABILITY = "Sustainability"


@dataclass
class Stakeholder:
    """A stakeholder group and its modeled concerns."""

    name: str
    type: str
    priority: float
    vulnerability: float
    geographic_context: str
    ethical_concerns: List[str] = field(default_factory=list)


@dataclass
class EthicalConstraint:
    """A minimum requirement measured on a specified scale."""

    principle: EthicalPrinciple
    description: str
    threshold: float
    weight: float
    applicability: List[str]
    measurement_method: str


@dataclass
class EthicalUtility:
    """A weighted utility function for comparing alternatives."""

    principle: EthicalPrinciple
    utility_function: Callable
    weight: float
    normalization_method: str = "min_max"


@dataclass
class DecisionAlternative:
    """A placement alternative with its calculated measures."""

    name: str
    description: str
    safety_score: float
    operational_cost: float
    ethical_scores: Dict[Union[EthicalPrinciple, str], float]
    stakeholder_impacts: Dict[str, float]
    ethical_compliance: bool
    reasoning: str
    confidence: float
    segregation_multiplier: float

In [ ]:
def create_ethical_framework_scenario():
    """Create the stakeholders, constraints, and utility functions."""
    stakeholders = [
        Stakeholder(
            "Crew Members", "crew", 0.9, 0.7, "mixed",
            ["personal_safety", "working_conditions", "exposure_to_hazards"],
        ),
        Stakeholder(
            "Marine Environment", "environment", 0.85, 0.9, "global",
            ["pollution_prevention", "ecosystem_protection", "biodiversity"],
        ),
        Stakeholder(
            "Southeast Asian Communities", "local_communities", 0.8, 0.9,
            "developing",
            ["environmental_justice", "health_protection", "economic_impact"],
        ),
        Stakeholder(
            "Shipping Line", "shipping_line", 0.7, 0.3, "developed",
            ["operational_efficiency", "cost_management", "regulatory_compliance"],
        ),
        Stakeholder(
            "Agricultural Importers", "cargo_owners", 0.6, 0.4, "developing",
            ["cargo_integrity", "delivery_reliability", "cost_control"],
        ),
        Stakeholder(
            "Port Authorities", "port_authorities", 0.7, 0.5, "mixed",
            ["safety_compliance", "operational_efficiency", "revenue_generation"],
        ),
    ]

    ethical_constraints = [
        EthicalConstraint(
            EthicalPrinciple.NON_MALEFICENCE,
            "The illustrative safety score must meet the minimum threshold.",
            0.95, 0.3, ["1.1", "1.4S", "6.1", "8", "8.1"],
            "safety_score",
        ),
        EthicalConstraint(
            EthicalPrinciple.JUSTICE,
            "The community justice utility must meet the minimum threshold.",
            0.8, 0.25, ["6.1", "8", "8.1"],
            "justice_utility",
        ),
        EthicalConstraint(
            EthicalPrinciple.PRECAUTIONARY,
            "The minimum relevant separation must be at least 120% of the assumed minimum.",
            1.2, 0.2, ["1.1", "1.4S", "6.1", "8.1"],
            "segregation_multiplier",
        ),
        EthicalConstraint(
            EthicalPrinciple.TRANSPARENCY,
            "A decision must include a substantive written rationale.",
            1.0, 0.15, ["all"],
            "explainability_score",
        ),
        EthicalConstraint(
            EthicalPrinciple.SUSTAINABILITY,
            "The illustrative sustainability utility must meet the minimum threshold.",
            0.7, 0.2, ["6.1", "8", "8.1"],
            "sustainability_utility",
        ),
    ]

    def non_maleficence_utility(decision):
        """A higher safety score represents less modeled risk of harm."""
        return decision.safety_score

    def beneficence_utility(decision):
        """Reward the modeled safety score."""
        return decision.safety_score

    def justice_utility(decision):
        """Reward smaller-magnitude modeled impacts on local communities."""
        impact = decision.stakeholder_impacts.get(
            "Southeast Asian Communities", 0.0
        )
        return 1.0 - min(1.0, abs(impact))

    def precautionary_utility(decision):
        """Reward separation that reaches the 120% precautionary target."""
        return min(1.0, decision.segregation_multiplier / 1.2)

    def transparency_utility(decision):
        """Give full credit to a substantive rationale of at least 100 characters."""
        return float(len(decision.reasoning.strip()) >= 100)

    def sustainability_utility(decision):
        """Combine normalized environmental impact and cost efficiency."""
        env_impact = decision.stakeholder_impacts.get("Marine Environment", 0.0)
        cost_efficiency = np.clip(1.0 - decision.operational_cost / 50000.0, 0.0, 1.0)
        return (1.0 - min(1.0, abs(env_impact))) * 0.6 + cost_efficiency * 0.4

    ethical_utilities = [
        EthicalUtility(EthicalPrinciple.NON_MALEFICENCE, non_maleficence_utility, 0.25),
        EthicalUtility(EthicalPrinciple.BENEFICENCE, beneficence_utility, 0.15),
        EthicalUtility(EthicalPrinciple.JUSTICE, justice_utility, 0.2),
        EthicalUtility(EthicalPrinciple.PRECAUTIONARY, precautionary_utility, 0.15),
        EthicalUtility(EthicalPrinciple.TRANSPARENCY, transparency_utility, 0.1),
        EthicalUtility(EthicalPrinciple.SUSTAINABILITY, sustainability_utility, 0.15),
    ]

    return stakeholders, ethical_constraints, ethical_utilities


stakeholders, ethical_constraints, ethical_utilities = create_ethical_framework_scenario()

print("Value-Aligned Ethical Framework")
print(f"  Stakeholders: {len(stakeholders)}")
print(f"  Ethical constraints: {len(ethical_constraints)}")
print(f"  Ethical utility functions: {len(ethical_utilities)}")

print("\nStakeholder analysis:")
for stakeholder in stakeholders:
    print(
        f"  {stakeholder.name} ({stakeholder.type}): "
        f"priority {stakeholder.priority:.2f}, "
        f"vulnerability {stakeholder.vulnerability:.2f}"
    )
    print(f"    Concerns: {', '.join(stakeholder.ethical_concerns)}")

print("\nEthical constraints:")
for constraint in ethical_constraints:
    print(
        f"  {constraint.principle.value}: threshold "
        f"{constraint.threshold:g} ({constraint.measurement_method})"
    )
    print(f"    {constraint.description}")

## 2. Define containers and candidate positions

Each container has an IMDG class, destination, economic value, and illustrative health and environmental risk inputs. Each position has a three-dimensional coordinate and sensitivity attributes. The coordinates permit explicit Euclidean distance calculations; the sensitivity attributes contribute to stakeholder-impact scores.

The placement alternatives below use these positions by index. Their separation and sensitivity measures are calculated from the data, rather than inferred from their names or descriptions.

In [ ]:
@dataclass
class Container:
    """A container and its illustrative risk attributes."""

    id: str
    imdg_class: str
    contents: str
    destination: str
    owner: str
    value: float
    environmental_risk: float
    health_risk: float


@dataclass
class Position:
    """A candidate stowage position and its modeled context."""

    id: str
    hold: int
    x: float
    y: float
    z: float
    proximity_to_population: float
    environmental_sensitivity: float
    structural_integrity: float


def create_concrete_example():
    """Create the example containers and candidate positions."""
    containers = [
        Container("PEST001", "6.1", "Agricultural Pesticides", "Southeast Asia", "AgriCorp", 85000, 0.7, 0.8),
        Container("PEST002", "6.1", "Herbicides", "Southeast Asia", "AgriCorp", 62000, 0.6, 0.7),
        Container("ACID001", "8.1", "Industrial Acids", "Southeast Asia", "ChemCo", 45000, 0.8, 0.9),
        Container("ACID002", "8.1", "Corrosive Cleaners", "Southeast Asia", "ChemCo", 38000, 0.5, 0.6),
    ]

    positions = [
        Position("H1_A1", 1, 0, 0, 0, 0.2, 0.3, 0.9),
        Position("H1_A2", 1, 4, 0, 0, 0.2, 0.3, 0.9),
        Position("H1_B1", 1, 0, 3, 0, 0.2, 0.3, 0.9),
        Position("H1_B2", 1, 4, 3, 0, 0.2, 0.3, 0.9),
        Position("H2_A1", 2, 0, 0, 5, 0.6, 0.7, 0.8),
        Position("H2_A2", 2, 4, 0, 5, 0.6, 0.7, 0.8),
        Position("H2_B1", 2, 0, 3, 5, 0.6, 0.7, 0.8),
        Position("H2_B2", 2, 4, 3, 5, 0.6, 0.7, 0.8),
        Position("H3_A1", 3, 0, 0, 10, 0.9, 0.9, 0.7),
        Position("H3_A2", 3, 4, 0, 10, 0.9, 0.9, 0.7),
        Position("H3_B1", 3, 0, 3, 10, 0.9, 0.9, 0.7),
        Position("H3_B2", 3, 4, 3, 10, 0.9, 0.9, 0.7),
    ]
    return containers, positions


containers, positions = create_concrete_example()

print("Concrete example: pesticides and industrial acids")
print(f"  Containers: {len(containers)}")
print(f"  Candidate positions: {len(positions)} (3 holds × 4 positions)")

print("\nContainer details:")
for container in containers:
    print(f"  {container.id}: {container.contents} (Class {container.imdg_class})")
    print(f"    Destination: {container.destination}; value: ${container.value:,.0f}")
    print(
        f"    Illustrative environmental risk: {container.environmental_risk:.0%}; "
        f"health risk: {container.health_risk:.0%}"
    )

print("\nAverage position sensitivity by hold:")
for hold in (1, 2, 3):
    hold_positions = [position for position in positions if position.hold == hold]
    average_proximity = np.mean([p.proximity_to_population for p in hold_positions])
    average_sensitivity = np.mean(
        [p.environmental_sensitivity for p in hold_positions]
    )
    print(
        f"  Hold {hold}: population proximity {average_proximity:.0%}; "
        f"environmental sensitivity {average_sensitivity:.0%}"
    )

## 3. Calculate alternative measures

The decision engine turns each placement into comparable measures. It calculates safety from pairwise distances, operational cost from hold assignments, and stakeholder impacts from position sensitivity and container attributes. It then evaluates each alternative against the thresholds.

For a set of placed containers \(i\), let \(d_{ij}\) be the Euclidean distance between positions \(i\) and \(j\), and let \(d_{\min}=3\) m be the illustrative minimum. The safety score is

$$
S=1-\frac{1}{|Q|}\sum_{(i,j)\in Q}\min\left(1,\frac{\max(0,d_{\min}-d_{ij})}{d_{\min}}\right),
$$

where \(Q\) contains each unordered pair of hazardous-class containers. If there are no such pairs, the score is defined as 1. The precautionary separation multiplier is the minimum pairwise distance divided by \(d_{\min}\). Thus, a value of 1.2 means the closest modeled pair is at least 120% of the assumed minimum.

These are deliberately simple example metrics. In particular, a geometric distance threshold alone is not an IMDG compatibility model.

In [ ]:
class ValueAlignedDecisionEngine:
    """Evaluate predefined placements with explicit ethical measures."""

    HAZARDOUS_CLASSES = {"6.1", "8", "8.1"}
    MINIMUM_DISTANCE = 3.0

    def __init__(self, stakeholders, ethical_constraints, ethical_utilities):
        self.stakeholders = stakeholders
        self.ethical_constraints = ethical_constraints
        self.ethical_utilities = ethical_utilities
        self.stakeholder_lookup = {s.name: s for s in stakeholders}

    @staticmethod
    def _distance(position_a, position_b):
        """Measure straight-line separation in the stated 3D coordinates."""
        return float(
            np.sqrt(
                (position_a.x - position_b.x) ** 2
                + (position_a.y - position_b.y) ** 2
                + (position_a.z - position_b.z) ** 2
            )
        )

    def _hazardous_pair_distances(self, container_placement, containers):
        """Return each unordered distance pair in the modeled hazardous set."""
        placed = [
            container for container in containers
            if container.imdg_class in self.HAZARDOUS_CLASSES
            and container.id in container_placement
        ]
        return [
            self._distance(
                container_placement[placed[i].id],
                container_placement[placed[j].id],
            )
            for i in range(len(placed))
            for j in range(i + 1, len(placed))
        ]

    def calculate_safety_score(self, container_placement, containers):
        """Score pairwise distance shortfalls on a scale from zero to one."""
        distances = self._hazardous_pair_distances(container_placement, containers)
        if not distances:
            return 1.0
        shortfalls = [
            min(
                1.0,
                max(0.0, self.MINIMUM_DISTANCE - distance)
                / self.MINIMUM_DISTANCE,
            )
            for distance in distances
        ]
        return float(np.clip(1.0 - np.mean(shortfalls), 0.0, 1.0))

    def calculate_segregation_multiplier(self, container_placement, containers):
        """Compare the closest modeled pair with the assumed minimum distance."""
        distances = self._hazardous_pair_distances(container_placement, containers)
        if not distances:
            return float("inf")
        return min(distances) / self.MINIMUM_DISTANCE

    def calculate_operational_cost(self, container_placement, containers):
        """Calculate the illustrative handling and placement costs."""
        base_cost = 25000
        handling_cost = sum(position.hold * 500 for position in container_placement.values())
        sensitive_hold_penalty = sum(
            2000
            for container in containers
            if container.id in container_placement
            and container_placement[container.id].hold == 3
            and container.imdg_class in self.HAZARDOUS_CLASSES
        )
        return float(base_cost + handling_cost + sensitive_hold_penalty)

    def calculate_stakeholder_impacts(self, container_placement, containers):
        """Calculate normalized illustrative impacts for the listed groups."""
        placed_containers = [
            container for container in containers
            if container.id in container_placement
        ]
        count = max(1, len(placed_containers))

        crew_risk = 0.0
        environmental_risk = 0.0
        community_risk = 0.0
        for container in placed_containers:
            position = container_placement[container.id]
            if container.imdg_class in self.HAZARDOUS_CLASSES:
                crew_risk += (
                    position.z / 15.0 * container.health_risk * 0.3
                )
                community_risk += (
                    position.proximity_to_population
                    * (container.health_risk + container.environmental_risk)
                    * 0.4
                )
            environmental_risk += (
                position.proximity_to_population
                * position.environmental_sensitivity
                * container.environmental_risk
                * 0.5
            )

        cost = self.calculate_operational_cost(container_placement, containers)
        impacts = {
            "Crew Members": -crew_risk / count,
            "Marine Environment": -environmental_risk / count,
            "Southeast Asian Communities": -community_risk / count,
            "Shipping Line": float(np.clip((30000 - cost) / 30000, -1.0, 1.0)),
            "Agricultural Importers": 1.0 - (len(placed_containers) / count) * 0.1,
            "Port Authorities": (len(placed_containers) / count) * 0.5,
        }
        return impacts

    def check_ethical_constraints(self, decision):
        """Test each constraint using a measure on its stated scale."""
        for constraint in self.ethical_constraints:
            if constraint.principle == EthicalPrinciple.NON_MALEFICENCE:
                measure = decision.safety_score
            elif constraint.principle == EthicalPrinciple.PRECAUTIONARY:
                measure = decision.segregation_multiplier
            else:
                measure = decision.ethical_scores.get(constraint.principle, 0.0)
            if measure < constraint.threshold:
                return False
        return True

    def calculate_ethical_scores(self, decision):
        """Evaluate and bound every configured ethical utility."""
        scores = {}
        for ethical_utility in self.ethical_utilities:
            utility = ethical_utility.utility_function(decision)
            scores[ethical_utility.principle] = float(np.clip(utility, 0.0, 1.0))
        return scores

    def generate_decision_alternatives(self, containers, positions):
        """Evaluate the three specified placement alternatives."""
        placement_specs = [
            (
                "Solution A - Cost Minimization",
                "Uses the lower-cost set of positions in Hold 1.",
                {"PEST001": 0, "PEST002": 1, "ACID001": 2, "ACID002": 3},
                "All containers use Hold 1 positions. The closest modeled pair is assessed from its coordinates; this alternative emphasizes the illustrative handling-cost measure.",
                0.8,
            ),
            (
                "Solution B - Enhanced Precautionary Spacing",
                "Places pesticides in Hold 2 and acids in Hold 3.",
                {"PEST001": 4, "PEST002": 5, "ACID001": 8, "ACID002": 9},
                "Pesticides are placed in Hold 2 and acids in Hold 3. The minimum modeled pairwise separation is calculated from the coordinates and compared with the assumed 3 m minimum; position sensitivity and operational cost are also included in the assessment.",
                0.9,
            ),
            (
                "Solution C - Balanced Approach",
                "Distributes the containers across the specified positions.",
                {"PEST001": 0, "PEST002": 4, "ACID001": 8, "ACID002": 2},
                "The placement distributes the containers across the holds. Safety, separation, stakeholder impacts, and cost are calculated using the same measures as for the other alternatives.",
                0.85,
            ),
        ]

        position_lookup = {position.id: position for position in positions}
        alternatives = []
        for name, description, position_indices, reasoning, confidence in placement_specs:
            placement = {
                container_id: positions[index]
                for container_id, index in position_indices.items()
            }
            if len(set(position.id for position in placement.values())) != len(placement):
                raise ValueError(f"{name} assigns more than one container to a position.")
            if set(placement) != {container.id for container in containers}:
                raise ValueError(f"{name} must assign every container exactly once.")
            if any(position.id not in position_lookup for position in placement.values()):
                raise ValueError(f"{name} uses a position outside the candidate set.")

            decision = DecisionAlternative(
                name=name,
                description=description,
                safety_score=self.calculate_safety_score(placement, containers),
                operational_cost=self.calculate_operational_cost(placement, containers),
                ethical_scores={},
                stakeholder_impacts=self.calculate_stakeholder_impacts(placement, containers),
                ethical_compliance=False,
                reasoning=reasoning,
                confidence=confidence,
                segregation_multiplier=self.calculate_segregation_multiplier(
                    placement, containers
                ),
            )
            decision.ethical_scores = self.calculate_ethical_scores(decision)
            decision.ethical_compliance = self.check_ethical_constraints(decision)
            alternatives.append(decision)
        return alternatives

## 4. Evaluate the alternatives

The next cell applies the same calculations to each supplied placement. A score of 1 indicates the best value on that particular normalized measure; a negative stakeholder impact indicates an adverse impact under the example's scoring assumptions. Constraint compliance is reported separately from the individual utility scores.

In [ ]:
decision_engine = ValueAlignedDecisionEngine(
    stakeholders, ethical_constraints, ethical_utilities
)
alternatives = decision_engine.generate_decision_alternatives(containers, positions)

print("IMDG segregation: illustrative value-aligned analysis")
print("=" * 68)
print("Evaluating predefined placements for pesticides and industrial acids.")
print("All risk and impact values are model scores, not operational measurements.")

for index, alternative in enumerate(alternatives, start=1):
    print(f"\nAlternative {index}: {alternative.name}")
    print(f"  Description: {alternative.description}")
    print(f"  Safety score: {alternative.safety_score:.3f}")
    print(
        "  Minimum separation multiplier: "
        f"{alternative.segregation_multiplier:.3f} × assumed minimum"
    )
    print(f"  Illustrative operational cost: ${alternative.operational_cost:,.0f}")
    print(f"  Constraint compliant: {alternative.ethical_compliance}")
    print(f"  Model confidence input: {alternative.confidence:.0%}")
    print("  Stakeholder impacts:")
    for stakeholder, impact in alternative.stakeholder_impacts.items():
        print(f"    {stakeholder}: {impact:+.3f}")
    print("  Ethical utility scores:")
    for principle, score in alternative.ethical_scores.items():
        print(f"    {principle.value}: {score:.3f}")
    print(f"  Rationale: {alternative.reasoning}")

print("\nConstraint assessment:")
for alternative in alternatives:
    status = "COMPLIANT" if alternative.ethical_compliance else "NON-COMPLIANT"
    print(f"  {alternative.name}: {status}")
    for constraint in ethical_constraints:
        if constraint.principle == EthicalPrinciple.NON_MALEFICENCE:
            measure = alternative.safety_score
        elif constraint.principle == EthicalPrinciple.PRECAUTIONARY:
            measure = alternative.segregation_multiplier
        else:
            measure = alternative.ethical_scores.get(constraint.principle, 0.0)
        if measure < constraint.threshold:
            print(
                f"    Fails {constraint.principle.value}: "
                f"{measure:.3f} < threshold {constraint.threshold:.3f}"
            )

## 5. Rank alternatives using weighted ethical utility

The weighted score is a normalized weighted average, so alternatives can be compared even when the utility functions have different conceptual meanings, provided they have been placed on the shared 0-to-1 scale. The weights express the model's chosen relative importance; they are not discovered by the algorithm.

The ranking step first restricts attention to alternatives that pass every constraint. If that set is empty, it ranks all supplied alternatives and states that no fully compliant option was found. The fallback makes the comparison possible but does not waive or satisfy any constraint.

In [ ]:
def ethical_optimization(decision_engine, alternatives):
    """Rank feasible alternatives by their normalized weighted utility score."""
    if not alternatives:
        raise ValueError("At least one decision alternative is required.")

    compliant_alternatives = [
        alternative for alternative in alternatives
        if alternative.ethical_compliance
    ]
    if compliant_alternatives:
        ranking_pool = compliant_alternatives
        print("All ranked alternatives satisfy the modeled constraints.")
    else:
        ranking_pool = alternatives
        print(
            "No fully compliant alternative was found. "
            "All supplied alternatives are ranked for comparison."
        )

    total_weight = sum(
        utility.weight for utility in decision_engine.ethical_utilities
    )
    if total_weight <= 0:
        raise ValueError("The total ethical utility weight must be positive.")

    for alternative in ranking_pool:
        weighted_score = sum(
            alternative.ethical_scores.get(utility.principle, 0.0)
            * utility.weight
            for utility in decision_engine.ethical_utilities
        ) / total_weight
        alternative.ethical_scores["WEIGHTED_ETHICAL"] = weighted_score

    ranked_alternatives = sorted(
        ranking_pool,
        key=lambda alternative: alternative.ethical_scores["WEIGHTED_ETHICAL"],
        reverse=True,
    )

    print("\nRanked alternatives:")
    for rank, alternative in enumerate(ranked_alternatives, start=1):
        print(
            f"  {rank}. {alternative.name}: "
            f"weighted score {alternative.ethical_scores['WEIGHTED_ETHICAL']:.3f}; "
            f"cost ${alternative.operational_cost:,.0f}"
        )

    best_alternative = ranked_alternatives[0]
    print(f"\nSelected for this model: {best_alternative.name}")
    print(
        "Weighted ethical score: "
        f"{best_alternative.ethical_scores['WEIGHTED_ETHICAL']:.3f}"
    )
    print(f"Illustrative operational cost: ${best_alternative.operational_cost:,.0f}")
    print(f"Passes every modeled constraint: {best_alternative.ethical_compliance}")

    baseline_cost = 30000
    additional_cost = best_alternative.operational_cost - baseline_cost
    assumed_incident_cost = 2300000
    illustrative_risk_fraction = best_alternative.safety_score * 0.5
    estimated_risk_value = assumed_incident_cost * illustrative_risk_fraction
    net_estimate = estimated_risk_value - additional_cost

    print("\nIllustrative cost-benefit calculation:")
    print(f"  Assumed baseline cost: ${baseline_cost:,.0f}")
    print(f"  Cost difference from baseline: ${additional_cost:,.0f}")
    print(f"  Assumed incident cost: ${assumed_incident_cost:,.0f}")
    print(f"  Applied illustrative risk fraction: {illustrative_risk_fraction:.3f}")
    print(f"  Estimated risk value: ${estimated_risk_value:,.0f}")
    print(f"  Arithmetic net estimate: ${net_estimate:,.0f}")
    print("  This scenario calculation is not a forecast or verified benefit.")
    return best_alternative


best_alternative = ethical_optimization(decision_engine, alternatives)

## 6. Visualize the decision measures

The plots provide complementary views: cost versus safety, constraint status, stakeholder impacts, principle-level utility scores, the illustrative cost-benefit calculation, and confidence inputs. The color scale in the impact matrix maps negative values to red, values near zero to yellow, and positive values to green. Confidence is an input attached to each alternative; it is not estimated from data by this notebook.

In [ ]:
def visualize_ethical_analysis(alternatives, best_alternative):
    """Display six views of the modeled alternative measures."""
    if not alternatives:
        raise ValueError("At least one alternative is required for visualization.")

    fig, axes = plt.subplots(2, 3, figsize=(19, 12))
    names = [alternative.name.replace("Solution ", "") for alternative in alternatives]
    costs = [alternative.operational_cost for alternative in alternatives]
    safety_scores = [alternative.safety_score for alternative in alternatives]

    ax = axes[0, 0]
    ax.scatter(costs, safety_scores, s=100, alpha=0.75, color="steelblue")
    for cost, score, name in zip(costs, safety_scores, names):
        ax.annotate(name, (cost, score), xytext=(5, 5), textcoords="offset points")
    best_index = alternatives.index(best_alternative)
    ax.scatter(
        costs[best_index], safety_scores[best_index], s=220,
        facecolors="none", edgecolors="black", linewidth=2,
        label="Selected alternative",
    )
    ax.set_xlabel("Illustrative operational cost ($)")
    ax.set_ylabel("Safety score (0–1)")
    ax.set_title("Safety and cost")
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[0, 1]
    colors = ["seagreen" if item.ethical_compliance else "indianred" for item in alternatives]
    bars = ax.bar(names, safety_scores, color=colors, alpha=0.8)
    ax.set_ylabel("Safety score (0–1)")
    ax.set_ylim(0, 1.15)
    ax.set_title("Safety score and constraint status")
    ax.grid(True, alpha=0.3, axis="y")
    for bar, item in zip(bars, alternatives):
        label = "Pass" if item.ethical_compliance else "Fail"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.03,
            label,
            ha="center",
            fontsize=8,
        )

    stakeholder_names = [
        "Crew", "Environment", "Communities", "Shipping line", "Importers", "Port"
    ]
    stakeholder_keys = [
        "Crew Members", "Marine Environment", "Southeast Asian Communities",
        "Shipping Line", "Agricultural Importers", "Port Authorities",
    ]
    impact_matrix = np.array([
        [item.stakeholder_impacts.get(key, 0.0) for key in stakeholder_keys]
        for item in alternatives
    ]).T
    image = axes[0, 2].imshow(
        impact_matrix, cmap="RdYlGn", aspect="auto", vmin=-1, vmax=1
    )
    axes[0, 2].set_xticks(range(len(names)), names, rotation=35, ha="right")
    axes[0, 2].set_yticks(range(len(stakeholder_names)), stakeholder_names)
    axes[0, 2].set_title("Illustrative stakeholder impacts")
    colorbar = fig.colorbar(image, ax=axes[0, 2])
    colorbar.set_label("Impact score (−1 to +1)")

    ax = axes[1, 0]
    principles = [utility.principle for utility in ethical_utilities]
    principle_labels = [principle.value for principle in principles]
    principle_scores = [
        best_alternative.ethical_scores.get(principle, 0.0)
        for principle in principles
    ]
    bars = ax.bar(principle_labels, principle_scores, color="skyblue", alpha=0.8)
    ax.set_ylabel("Utility score (0–1)")
    ax.set_ylim(0, 1.15)
    ax.set_title(f"Principle scores: {best_alternative.name}")
    ax.tick_params(axis="x", labelrotation=40)
    ax.grid(True, alpha=0.3, axis="y")
    for bar, score in zip(bars, principle_scores):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{score:.2f}",
            ha="center",
            fontsize=8,
        )

    ax = axes[1, 1]
    baseline_cost = 30000
    additional_costs = [item.operational_cost - baseline_cost for item in alternatives]
    risk_values = [item.safety_score * 2300000 * 0.5 for item in alternatives]
    net_estimates = [risk - cost for risk, cost in zip(risk_values, additional_costs)]
    x_positions = np.arange(len(names))
    width = 0.25
    ax.bar(x_positions - width, additional_costs, width, label="Cost difference", color="firebrick")
    ax.bar(x_positions, risk_values, width, label="Illustrative risk value", color="seagreen")
    ax.bar(x_positions + width, net_estimates, width, label="Arithmetic net estimate", color="steelblue")
    ax.set_xlabel("Alternative")
    ax.set_ylabel("Illustrative value ($)")
    ax.set_title("Scenario cost-benefit arithmetic")
    ax.set_xticks(x_positions, names, rotation=30, ha="right")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")

    ax = axes[1, 2]
    for index, item in enumerate(alternatives):
        color = "seagreen" if item.ethical_compliance else "indianred"
        size = 100 + item.confidence * 200
        ax.scatter(index, item.confidence, s=size, c=color, alpha=0.75, edgecolors="black")
        ax.annotate(names[index], (index, item.confidence), xytext=(0, 9),
                    textcoords="offset points", ha="center", fontsize=8)
    ax.set_xlabel("Alternative index")
    ax.set_ylabel("Supplied confidence input (0–1)")
    ax.set_title("Confidence input and constraint status")
    ax.set_xticks(range(len(names)), [str(i + 1) for i in range(len(names))])
    ax.set_ylim(0, 1.15)
    ax.grid(True, alpha=0.3, axis="y")
    ax.scatter([], [], s=100, color="seagreen", label="Passes constraints")
    ax.scatter([], [], s=100, color="indianred", label="Fails a constraint")
    ax.legend(fontsize=8)

    fig.suptitle("Illustrative IMDG Ethical Decision Analysis", fontsize=15)
    fig.tight_layout()
    plt.show()


visualize_ethical_analysis(alternatives, best_alternative)

## 7. Generate a decision report

The report consolidates the selected alternative, its measures, constraint results, and the scenario arithmetic. It distinguishes modeled outputs from regulatory conclusions: the simplified example cannot establish that IMDG requirements or other legal obligations are satisfied.

In [ ]:
def generate_ethical_decision_report(best_alternative, decision_engine):
    """Print a reproducible summary of the selected modeled alternative."""
    print("\n" + "=" * 76)
    print("ILLUSTRATIVE ETHICAL DECISION REPORT")
    print("=" * 76)
    print(f"\nSelected alternative: {best_alternative.name}")
    print(f"Model confidence input: {best_alternative.confidence:.1%}")
    print(
        f"The evaluation includes {len(decision_engine.ethical_constraints)} "
        f"constraints and {len(decision_engine.ethical_utilities)} utility functions."
    )
    if best_alternative.ethical_compliance:
        print("The alternative passes all constraints represented in this model.")
    else:
        print("The alternative does not pass every constraint represented in this model.")

    print("\nModeled measures:")
    print(f"  Safety score: {best_alternative.safety_score:.3f} (0–1)")
    print(
        "  Minimum separation multiplier: "
        f"{best_alternative.segregation_multiplier:.3f} × assumed minimum"
    )
    print(f"  Illustrative operational cost: ${best_alternative.operational_cost:,.0f}")
    print("\nEthical utility scores:")
    for principle, score in best_alternative.ethical_scores.items():
        if isinstance(principle, EthicalPrinciple):
            print(f"  {principle.value}: {score:.3f}")

    print("\nStakeholder impact scores:")
    for stakeholder, impact in best_alternative.stakeholder_impacts.items():
        direction = "positive" if impact > 0.1 else "negative" if impact < -0.1 else "near neutral"
        print(f"  {stakeholder}: {direction} ({impact:+.3f})")

    print("\nConstraint assessment:")
    for constraint in decision_engine.ethical_constraints:
        if constraint.principle == EthicalPrinciple.NON_MALEFICENCE:
            measure = best_alternative.safety_score
        elif constraint.principle == EthicalPrinciple.PRECAUTIONARY:
            measure = best_alternative.segregation_multiplier
        else:
            measure = best_alternative.ethical_scores.get(constraint.principle, 0.0)
        status = "PASS" if measure >= constraint.threshold else "FAIL"
        print(
            f"  {constraint.principle.value}: {status}; "
            f"measure {measure:.3f}, threshold {constraint.threshold:.3f}"
        )

    baseline_cost = 30000
    additional_cost = best_alternative.operational_cost - baseline_cost
    assumed_incident_cost = 2300000
    risk_value = assumed_incident_cost * best_alternative.safety_score * 0.5
    net_estimate = risk_value - additional_cost
    print("\nIllustrative cost-benefit arithmetic:")
    print(f"  Assumed baseline cost: ${baseline_cost:,.0f}")
    print(f"  Cost difference: ${additional_cost:,.0f}")
    print(f"  Estimated risk value under stated assumptions: ${risk_value:,.0f}")
    print(f"  Arithmetic net estimate: ${net_estimate:,.0f}")
    print("  This estimate is not a validated forecast or realized economic benefit.")

    print("\nDecision rationale:")
    print(f"  {best_alternative.reasoning}")
    print("\nInterpretation and use:")
    print("  Treat these scores as outputs of the stated illustrative assumptions.")
    print("  Verify compatibility, distances, and legal requirements independently.")
    print("  Document relevant operational evidence and stakeholder input separately.")
    print("\nThis report is not an IMDG compliance certificate or stowage instruction.")
    print("=" * 76)


generate_ethical_decision_report(best_alternative, decision_engine)

## 8. Interpretation, strengths, and limitations

### What this framework contributes

- It makes several stakeholder groups and ethical principles explicit in the decision record.
- It distinguishes hard minimum thresholds from weighted preferences.
- It provides interpretable intermediate measures, so a reader can inspect why an alternative passes or fails a modeled constraint.
- It makes trade-offs visible by reporting costs, safety scores, stakeholder impacts, and principle utilities separately.

### What the model does not establish

- The alternatives are predefined, so the selected alternative is best only within the ranking pool and scoring assumptions; it is not a globally optimal stowage plan.
- The distance rule is a simplified teaching assumption, not a substitute for class-specific IMDG segregation rules, packing-group details, or competent operational review.
- Utility weights and thresholds encode judgments. Different evidence, stakeholder consultation, or policy priorities can change the ranking.
- A weighted average can allow a strong score on one principle to offset a weaker score on another. The separate constraints limit this effect only for the specific thresholds that are modeled.
- The stakeholder scores and cost-benefit arithmetic are illustrative and should not be interpreted as measured impacts, causal estimates, or proof of ethical compliance.

### Appropriate use

This type of framework is useful for structured scenario discussion when decision-makers need to record assumptions, compare specified options, and expose value trade-offs. In operational use, the input data, constraint definitions, regulatory interpretation, risk measures, and stakeholder assessments require validation by qualified specialists and affected stakeholders. Additional scenarios or constraints may be studied within the same explicit scoring-and-screening structure, but the numerical example here should not be generalized beyond its stated assumptions.